# Assignment 3 — recover and continue

Select CPU and Runtime > Run all. Enter your existing Free Tier key privately. Restores 50 Flash-Lite results and six genuine Flash results. Failed requests are deferred so later invoices can run; up to three passes retry unfinished invoices. Three consecutive failures pause a pass to avoid repeatedly hitting an unavailable service. Quota/access errors stop calls. No paid upgrade or model substitution is used. Progress downloads even when the service remains unavailable. Refinement and the final report require all 50 Flash results. Download your executed notebook separately via File > Download > Download .ipynb. Source text is sent to Google; ground truth is only used for evaluation. Use your project with billing disabled.


In [ ]:
import subprocess,sys
subprocess.run([sys.executable,'-m','pip','install','pypdf>=5,<7','requests'],check=True)
print('Ready. No GPU or large model download is needed.')


In [ ]:
from pathlib import Path
import io,json,zipfile,requests,os,hashlib,datetime,importlib.metadata,shutil,csv,getpass
WORK=Path('/content/assignment3_recovery_run');WORK.mkdir(exist_ok=True)
r=requests.get('https://codeload.github.com/MVEST22/invoice-extraction-assignment/zip/refs/heads/main',timeout=120)
r.raise_for_status()
with zipfile.ZipFile(io.BytesIO(r.content)) as z:
    for entry in z.infolist():
        p=Path(entry.filename)
        if p.is_absolute() or '..' in p.parts:raise ValueError('Unexpected source archive path')
    z.extractall(WORK)
ROOT=WORK/'invoice-extraction-assignment-main';os.chdir(ROOT)
subprocess.run([sys.executable,'scripts/setup_data.py'],check=True)
subprocess.run([sys.executable,'scripts/preflight.py'],check=True)
sys.path.insert(0,str(ROOT/'methods'))
print('50 source invoices ready; answers are reserved for evaluation.')

# Restore evidence, never overwrite newer local progress or source code.
with zipfile.ZipFile(ROOT/'runs/2026-11-02_partial/checkpoint.zip') as z:
    for entry in z.infolist():
        if entry.is_dir():continue
        parts=Path(entry.filename).parts
        if not parts or parts[0]!='outputs' or '..' in parts:continue
        target=ROOT/entry.filename
        # Repair empty files created for ZIP directory entries by the previous template.
        for parent in reversed(target.parents):
            if parent==ROOT or ROOT in parent.parents:
                if parent.is_file():
                    backup=parent.with_name(parent.name+'.old_zip_directory_entry')
                    if backup.exists():raise RuntimeError('Unexpected backup collision: '+str(backup))
                    parent.rename(backup)
                parent.mkdir(exist_ok=True)
        if not target.exists():
            target.parent.mkdir(parents=True,exist_ok=True)
            target.write_bytes(z.read(entry))
manifest=json.loads((ROOT/'runs/2026-11-02_partial/run_manifest.json').read_text())
for name,digest in manifest['input_sha256'].items():
    assert hashlib.sha256((ROOT/'invoices'/name).read_bytes()).hexdigest()==digest
print('Saved results restored. Method 2 will make no new API requests.')

# Restore the sixth genuine response from the latest returned checkpoint.
seed=ROOT/'runs/2026-11-02_partial/extra_raw'
for source in seed.glob('*.json'):
    target=ROOT/'outputs/raw/method3_expensive'/source.name
    if not target.exists():shutil.copyfile(source,target)


In [ ]:
subprocess.run([sys.executable,'methods/method1_programmatic.py'],check=True)
subprocess.run([sys.executable,'evaluation/compare_results.py'],check=True)


In [ ]:
# Explicit reload avoids stale code if you rerun cells in an existing session.
import importlib,gemini_extract
importlib.reload(gemini_extract)
MODEL_PAIR={'method2_low_cost':'gemini-3.5-flash-lite','method3_expensive':'gemini-3.8-flash'}
KEY=getpass.getpass('Gemini Free Tier key (private): ').strip()
if not KEY:raise RuntimeError('Enter your key in the private prompt.')


In [ ]:
RUN_ERROR=None
try:
    for recovery_pass in range(1,4):
        print(f'Recovery pass {recovery_pass}/3. Completed invoices are reused.')
        try:
            gemini_extract.run(KEY,'method3_expensive',MODEL_PAIR['method3_expensive'],batch_size=1,continue_transient=True)
            RUN_ERROR=None
            break
        except RuntimeError as exc:
            RUN_ERROR=str(exc)
            print('Pass paused:',RUN_ERROR)
            if 'HTTP 429' in RUN_ERROR or 'HTTP 400' in RUN_ERROR or 'HTTP 403' in RUN_ERROR:break
            if recovery_pass<3:
                print('Retrying only unfinished invoices in 20 seconds.')
                import time
                time.sleep(20)
    if RUN_ERROR:print('Still incomplete. Progress will download with pending invoices listed.')
finally:
    KEY=None


In [ ]:
def rows(method):
    p=ROOT/'outputs'/(method+'.csv')
    return len(list(csv.DictReader(p.open()))) if p.exists() else 0
subprocess.run([sys.executable,'evaluation/compare_results.py'],check=True)
if rows('method3_expensive')==50:
    shutil.copyfile(ROOT/'evaluation/summary.json',ROOT/'evaluation/initial_three_methods.json')
    for script in ['methods/method4_refined.py','evaluation/compare_results.py','scripts/calculate_costs.py','scripts/build_db.py','scripts/build_report.py']:
        subprocess.run([sys.executable,script],check=True)
    print((ROOT/'REPORT.md').read_text())
else:
    print('Method 3 incomplete. Final comparison and refinement wait for all 50 invoices.')
    subprocess.run([sys.executable,'scripts/calculate_costs.py'],check=True)


In [ ]:
from google.colab import files
DEST=WORK/'results_package';DEST.mkdir(exist_ok=True)
for directory in ['outputs','evaluation','methods','scripts']:
    shutil.copytree(ROOT/directory,DEST/directory,dirs_exist_ok=True,ignore=shutil.ignore_patterns('__pycache__'))
for name in ['REPORT.md','README.md','requirements.txt','run_pipeline.py','tests_local.py','invoice_results.db','Assignment_3_Recovery_Colab.ipynb']:
    if (ROOT/name).exists():shutil.copyfile(ROOT/name,DEST/name)
status={}
for method in ['method1_programmatic','method2_low_cost','method3_expensive','method4_refined']:
    path=ROOT/'outputs'/(method+'.csv')
    if path.exists():
        with path.open(encoding='utf-8-sig',newline='') as f:status[method]=len(list(csv.DictReader(f)))
    else:status[method]=0
costpath=ROOT/'evaluation/costs.json'
costs=json.loads(costpath.read_text()) if costpath.exists() else []
manifest={'packaged_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),'rows':status,
    'complete':all(n==50 for n in status.values()),'method':'Gemini API on CPU Colab; no local model inference',
    'models':globals().get('MODEL_PAIR',{}),'billing_basis':'Eligible Free Tier project chosen by user; billing disabled; not independently verified by inference API.',
    'free_tier_charge_usd':0,'usage':costs,
    'packages':{p:importlib.metadata.version(p) for p in ['pypdf','requests']},
    'input_sha256':{p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted((ROOT/'invoices').glob('*.pdf'))},
    'code_sha256':{str(p.relative_to(ROOT)):hashlib.sha256(p.read_bytes()).hexdigest() for d in ['methods','scripts','evaluation'] for p in (ROOT/d).glob('*.py')}}
(DEST/'run_manifest.json').write_text(json.dumps(manifest,indent=2))
(DEST/'EXECUTION_NOTE.txt').write_text('The included notebook is an unexecuted source template. Download your actual executed notebook separately using File > Download > Download .ipynb. Return that notebook and this ZIP. Check manifest complete and row counts. API key is not included.')
archive=shutil.make_archive(str(WORK/'assignment3_results'),'zip',DEST)
print('Rows per method:',status,'Complete:',manifest['complete'])
files.download(archive)
